In [7]:
import pandas as pd
from pathlib import Path

FILE    = Path("23in14ar.xls")
OUT_DIR = Path("../public")

# Load
df = pd.read_excel(FILE, engine="xlrd", header=None, skiprows=8)
df.columns = range(len(df.columns))


In [8]:

# Extract columns — filter before rename
df_sub = df[[0, 1, 141, 142, 147, 148]].copy()
df_sub = df_sub[df_sub[0].notna()].copy()
df_sub.columns = ["agi_bracket", "num_returns", "n_taxable_income", "taxable_income_thousands", "n_income_tax", "income_tax_thousands"]

for col in ["num_returns", "n_taxable_income", "taxable_income_thousands", "n_income_tax", "income_tax_thousands"]:
    df_sub[col] = pd.to_numeric(df_sub[col], errors="coerce")

# Verify totals
all_row    = df_sub[df_sub["agi_bracket"] == "All returns, total"].iloc[0]
total_ti_b = all_row["taxable_income_thousands"] / 1e6
total_tx_b = all_row["income_tax_thousands"] / 1e6
print(f"Total taxable income:   ${total_ti_b:.2f}T  (expect ~$11.9T)")
print(f"Total income tax:       ${total_tx_b:.2f}T  (expect ~$2.27T)")
print(f"Implied effective rate: {total_tx_b/total_ti_b*100:.1f}%")
print()

# Keep only first block of AGI brackets (before "Taxable returns, total")
SKIP = ["All returns, total", "Taxable returns, total", "Nontaxable returns, total", "No adjusted gross income"]
first_block_end = df_sub[df_sub["agi_bracket"] == "Taxable returns, total"].index[0]
brackets = df_sub[~df_sub["agi_bracket"].isin(SKIP) & (df_sub.index < first_block_end)].copy()
brackets = brackets[brackets["taxable_income_thousands"].notna() & (brackets["taxable_income_thousands"] > 0)].copy()

brackets["taxable_income_b"]     = brackets["taxable_income_thousands"] / 1e6
brackets["income_tax_b"]         = brackets["income_tax_thousands"] / 1e6
brackets["effective_rate_pct"]   = (brackets["income_tax_b"] / brackets["taxable_income_b"] * 100).round(1)
brackets["revenue_per_1pp_b"]    = (brackets["taxable_income_b"] * 0.01).round(2)
brackets["num_returns_millions"] = (brackets["num_returns"] / 1e6).round(2)

result = brackets[["agi_bracket","num_returns_millions","taxable_income_b","income_tax_b","effective_rate_pct","revenue_per_1pp_b"]].reset_index(drop=True)
print(result.to_string(index=False))
print()


Total taxable income:   $11944.45T  (expect ~$11.9T)
Total income tax:       $2272.91T  (expect ~$2.27T)
Implied effective rate: 19.0%

                 agi_bracket  num_returns_millions  taxable_income_b  income_tax_b  effective_rate_pct  revenue_per_1pp_b
             $1 under $5,000                  7.36          0.220041      0.049503                22.5               0.00
        $5,000 under $10,000                  8.08          0.471500      0.083507                17.7               0.00
       $10,000 under $15,000                  8.99          1.236564      0.196144                15.9               0.01
       $15,000 under $20,000                  8.70         18.741676      1.938059                10.3               0.19
       $20,000 under $25,000                  7.93         44.409930      4.463819                10.1               0.44
       $25,000 under $30,000                  7.59         74.527042      7.753236                10.4               0.75
       $30

In [12]:

# Sanity checks
print("--- Sanity checks ---")
checks = [
    ("$1 under $5,000",           "taxable_income_b",    0,      1),
    ("$75,000 under $100,000",    "taxable_income_b",    800,    1200),
    ("$100,000 under $200,000",   "taxable_income_b",    2000,   4000),
    ("$200,000 under $500,000",   "taxable_income_b",    2000,   3500),
    ("$10,000,000 or more",       "taxable_income_b",    500,    1000),
]
all_ok = True
for bracket, col, lo, hi in checks:
    row = result[result["agi_bracket"] == bracket]
    if row.empty:
        print(f"  MISSING {bracket}"); all_ok = False; continue
    val = row[col].values[0]
    ok  = lo <= val <= hi
    print(f"  {'OK  ' if ok else 'FAIL'}  {bracket}: {col} = {val:.2f}  (expect {lo}-{hi})")
    if not ok: all_ok = False
print("All checks passed ✓" if all_ok else "Some checks FAILED.")
print()


--- Sanity checks ---
  OK    $1 under $5,000: taxable_income_b = 0.22  (expect 0-1)
  OK    $75,000 under $100,000: taxable_income_b = 1001.46  (expect 800-1200)
  OK    $100,000 under $200,000: taxable_income_b = 3055.40  (expect 2000-4000)
  OK    $200,000 under $500,000: taxable_income_b = 2760.66  (expect 2000-3500)
  OK    $10,000,000 or more: taxable_income_b = 804.25  (expect 500-1000)
All checks passed ✓



In [13]:

# Bucket and write
BUCKET_MAP = {
    "$1 under $5,000":                "Under $25K",
    "$5,000 under $10,000":           "Under $25K",
    "$10,000 under $15,000":          "Under $25K",
    "$15,000 under $20,000":          "Under $25K",
    "$20,000 under $25,000":          "Under $25K",
    "$25,000 under $30,000":          "$25K to $75K",
    "$30,000 under $40,000":          "$25K to $75K",
    "$40,000 under $50,000":          "$25K to $75K",
    "$50,000 under $75,000":          "$25K to $75K",
    "$75,000 under $100,000":         "$75K to $200K",
    "$100,000 under $200,000":        "$75K to $200K",
    "$200,000 under $500,000":        "$200K to $1M",
    "$500,000 under $1,000,000":      "$200K to $1M",
    "$1,000,000 under $1,500,000":    "Over $1M",
    "$1,500,000 under $2,000,000":    "Over $1M",
    "$2,000,000 under $5,000,000":    "Over $1M",
    "$5,000,000 under $10,000,000":   "Over $1M",
    "$10,000,000 or more":            "Over $1M",
}

result["bucket"] = result["agi_bracket"].map(BUCKET_MAP)
result = result[result["bucket"].notna()].copy()

ORDER = ["Under $25K", "$25K to $75K", "$75K to $200K", "$200K to $1M", "Over $1M"]
bucketed = result.groupby("bucket", sort=False).agg(
    num_returns_millions = ("num_returns_millions", "sum"),
    taxable_income_b     = ("taxable_income_b",     "sum"),
    income_tax_b         = ("income_tax_b",         "sum"),
    revenue_per_1pp_b    = ("revenue_per_1pp_b",    "sum"),
).reset_index()

bucketed["effective_rate_pct"] = (bucketed["income_tax_b"] / bucketed["taxable_income_b"] * 100).round(1)
bucketed["revenue_per_1pp_b"]  = bucketed["revenue_per_1pp_b"].round(1)
bucketed["order"]              = bucketed["bucket"].map({b: i for i, b in enumerate(ORDER)})
bucketed                       = bucketed.sort_values("order").drop(columns="order")

print("--- Bucketed output ---")
print(bucketed.to_string(index=False))


--- Bucketed output ---
       bucket  num_returns_millions  taxable_income_b  income_tax_b  revenue_per_1pp_b  effective_rate_pct
   Under $25K                 41.06         65.079711      6.731032                0.6                10.3
 $25K to $75K                 60.46       1683.256657    192.829507               16.8                11.5
$75K to $200K                 43.38       4056.863683    587.764430               40.6                14.5
 $200K to $1M                 12.74       3852.500039    806.066109               38.5                20.9
     Over $1M                  0.80       2286.746902    679.374470               22.9                29.7


In [14]:
out = OUT_DIR / "tax_brackets.csv"
bucketed.to_csv(out, index=False)
print(f"\nWritten -> {out}")


Written -> ../public/tax_brackets.csv
